# YOLOv2 — Training on Your Own Dataset (darkflow)

Train YOLOv2 on a custom class, then detect on **images**, a **YouTube video**,
and the **webcam**. This follows the course lesson step for step.

The course ran on Windows with TF 1.11 + CUDA, with the working directory set to
`darkflow/`. Only what had to change for this machine has changed:

| Course | Here |
| --- | --- |
| cwd `darkflow/`, paths like `cfg/yolov2-1c.cfg` | same files, addressed under `custom/` (this notebook sits in the module root) |
| `'gpu': 1.0` on CUDA | `'gpu': 0.8` on Apple Silicon via `tensorflow-metal` |
| `pafy` + `youtube_dl` | the same cell, plus a `yt-dlp` version — see the video section |

The command-line equivalent is [yolov2_custom_object.py](yolov2_custom_object.py),
written up in [19.Custom-Object-Detection-with-Yolo.md](19.Custom-Object-Detection-with-Yolo.md).

## Setup

In [1]:
%matplotlib inline
import os
import sys

# darkflow is vendored (not pip-installed), so put its repo root on sys.path.
DARKFLOW_DIR = os.path.abspath('./darkflow')
if DARKFLOW_DIR not in sys.path:
    sys.path.insert(0, DARKFLOW_DIR)

# The vendored sources are patched for TF2 / NumPy 2. If this kernel already
# imported an older copy, Python would keep serving it from memory.
for _mod in [m for m in sys.modules if m == 'darkflow' or m.startswith('darkflow.')]:
    del sys.modules[_mod]

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import matplotlib.pyplot as plt
import numpy as np
import cv2
from darkflow.net.build import TFNet

print('OpenCV:', cv2.__version__)

OpenCV: 5.0.0


## The dataset and the model files

The course prepares this by hand: copy `tiny-yolo-voc.cfg` to a `-1c` name, edit
`classes` and the last `filters`, write `labels.txt`, and drop the images and XML
annotations into `train/`. `prepare()` does exactly that, and is safe to re-run.

Two details it takes care of:

- **The filters formula.** The last convolutional layer needs
  `5 x (N_classes + 5)` filters — **30** for one class, against 425 for COCO's
  80. `prepare` checks `classes=` and the final `filters=` against the labels.
- **`cfg/yolov2.cfg` must sit next to `cfg/yolov2-1c.cfg`.** Not in the course
  notes: when `load` is a `.weights` path, darkflow looks for a cfg of the *same
  basename* to know the layout to read the binary with. Without it, darkflow
  falls back to the 1-class cfg and dies on a byte-count assertion, because the
  file holds 425-filter weights.

In [2]:
# Reuse the CLI script's setup so there is one source of truth for the layout.
import yolov2_custom_object as yco

yco.prepare()

MODEL_CFG = yco.MODEL_CFG        # custom/cfg/yolov2-1c.cfg
CFG_DIR   = yco.CFG_DIR          # holds yolov2-1c.cfg AND yolov2.cfg
LABELS    = yco.LABELS_TXT       # one line: r2d2
CKPT_DIR  = yco.CKPT_DIR         # custom/ckpt
IMAGES    = yco.TRAIN_IMAGES     # custom/train/images
ANNOTS    = yco.TRAIN_ANNOTATIONS
WEIGHTS   = yco.SRC_WEIGHTS      # resources/14.4 yolov2_weights/yolov2.weights

# Metal GPU fraction. Set to 0.0 to run on CPU.
GPU = 0.8

cfg      : /Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/54.Deep-Learning-for-Computer-Vision-with-TensorFlow-2/custom/cfg/yolov2-1c.cfg (+ yolov2.cfg source arch)
labels   : /Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/54.Deep-Learning-for-Computer-Vision-with-TensorFlow-2/custom/labels.txt -> r2d2
images   : ok
annots   : ok
dataset  : 200 images, 200 annotations
cfg check: classes=1, last filters=30 (5 x (1 + 5))


## Train the model

`load` is the pretrained COCO `yolov2.weights`: every layer but the last is
transferred and the mismatched final layer is randomly initialised — the transfer
learning this lesson is about.

**This takes hours.** The course runs 100 epochs and reaches loss ~0.68 at step
1200; 300 epochs was already run from the command line here, so `custom/ckpt/`
is populated and you can skip straight to the next section if you just want to
see detections.

In [ ]:
options = {"model": MODEL_CFG,
           "load": WEIGHTS,
           "config": CFG_DIR,
           "labels": LABELS,
           "backup": CKPT_DIR,
           "epoch": 10,
           "gpu": GPU,
           "train": True,
           "annotation": ANNOTS,
           "dataset": IMAGES}

tfnet = TFNet(options)

Parsing /Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/54.Deep-Learning-for-Computer-Vision-with-TensorFlow-2/custom/cfg/yolov2.cfg
Parsing /Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/54.Deep-Learning-for-Computer-Vision-with-TensorFlow-2/custom/cfg/yolov2-1c.cfg
Loading /Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/54.Deep-Learning-for-Computer-Vision-with-TensorFlow-2/resources/14.4 yolov2_weights/yolov2.weights ...
Successfully identified 203934260 bytes
Finished in 0.012191057205200195s

Building net ...
Source | Train? | Layer description                | Output size
-------+--------+----------------------------------+---------------
       |        | input                            | (None, 416, 416, 3)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (None, 416, 416, 32)
 Load  |  Yep!  | maxp 2x2p0_2                     | (None, 208, 208, 32)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (None, 208, 208, 64)
 Load  |  Yep!  | maxp 2x2p0_2   

I0000 00:00:1788714114.173710  423979 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1788714114.173761  423979 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


 Load  |  Yep!  | conv 1x1p0_1  +bnorm  leaky      | (None, 52, 52, 128)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (None, 52, 52, 256)
 Load  |  Yep!  | maxp 2x2p0_2                     | (None, 26, 26, 256)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (None, 26, 26, 512)
 Load  |  Yep!  | conv 1x1p0_1  +bnorm  leaky      | (None, 26, 26, 256)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (None, 26, 26, 512)
 Load  |  Yep!  | conv 1x1p0_1  +bnorm  leaky      | (None, 26, 26, 256)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (None, 26, 26, 512)
 Load  |  Yep!  | maxp 2x2p0_2                     | (None, 13, 13, 512)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (None, 13, 13, 1024)
 Load  |  Yep!  | conv 1x1p0_1  +bnorm  leaky      | (None, 13, 13, 512)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (None, 13, 13, 1024)
 Load  |  Yep!  | conv 1x1p0_1  +bnorm  leaky      | (None, 13, 13, 512)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky    

I0000 00:00:1788714116.983326  423979 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1788714116.983370  423979 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
I0000 00:00:1788714117.197442  423979 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


Finished in 6.206655979156494s



In [4]:
%%time
tfnet.train()


/Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/54.Deep-Learning-for-Computer-Vision-with-TensorFlow-2/custom/cfg/yolov2-1c.cfg parsing /Users/ankit/Workspace/Projects/ankit-github/AI-ML-DS/54.Deep-Learning-for-Computer-Vision-with-TensorFlow-2/custom/train/annotations
Parsing for ['r2d2'] 
[====================>]100%  000188.xml
Statistics:
r2d2: 235
Dataset size: 200
Dataset of 200 instance(s)
Training statistics: 
	Learning rate : 1e-05
	Batch size    : 16
	Epoch number  : 100
	Backup every  : 2000
step 1 - loss 104.6254653930664 - moving ave loss 104.6254653930664
step 2 - loss 103.3206787109375 - moving ave loss 104.49498748779297
step 3 - loss 102.93272399902344 - moving ave loss 104.33876037597656
step 4 - loss 102.04838562011719 - moving ave loss 104.10972595214844
step 5 - loss 101.8631362915039 - moving ave loss 103.88507080078125
step 6 - loss 101.16444396972656 - moving ave loss 103.61300659179688
step 7 - loss 100.68714904785156 - moving ave loss 103.32041931152344


KeyboardInterrupt: 

## Load the trained model

`"load": -1` picks the most recent checkpoint. Pass a step number instead
(the course uses `1200`) to pin one.

In [ ]:
options = {"model": MODEL_CFG,
           "config": CFG_DIR,
           "labels": LABELS,
           "backup": CKPT_DIR,
           "threshold": 0.1,
           "load": -1,
           "gpu": GPU}

tfnet2 = TFNet(options)

In [ ]:
tfnet2.load_from_ckpt()

## Object detection on images

`return_predict` gives a list of dicts: `label`, `confidence`, and the `topleft`
/ `bottomright` corners.

In [ ]:
original_img = cv2.imread(os.path.join(IMAGES, "000004.jpg"))
original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
results = tfnet2.return_predict(original_img)
print(results)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(original_img)

### Draw the boxes

A red rectangle plus a green italic `label confidence`. The image is in RGB
here, so `(255, 0, 0)` is red and `(0, 255, 0)` is green.

In [ ]:
def plot_box(original_img, predictions):
    newImage = np.copy(original_img)

    for result in predictions:
        top_x = result['topleft']['x']
        top_y = result['topleft']['y']

        btm_x = result['bottomright']['x']
        btm_y = result['bottomright']['y']

        confidence = result['confidence'] * 100
        label = result['label'] + " " + str(round(confidence, 2))

        newImage = cv2.rectangle(newImage, (top_x, top_y), (btm_x, btm_y), (255, 0, 0), 5)
        newImage = cv2.putText(newImage, label, (top_x, top_y - 5), cv2.FONT_ITALIC, 1, (0, 255, 0), 2)

    return newImage

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(plot_box(original_img, results))

### A few more images

In [ ]:
samples = ['000000.jpg', '000005.jpg', '000050.jpg', '000120.jpg']

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for ax, name in zip(axes.ravel(), samples):
    img = cv2.cvtColor(cv2.imread(os.path.join(IMAGES, name)), cv2.COLOR_BGR2RGB)
    preds = tfnet2.return_predict(img)
    ax.imshow(plot_box(img, preds))
    ax.set_title('{}  ({} detection(s))'.format(name, len(preds)))
    ax.axis('off')

plt.tight_layout()
plt.show()

## Object detection on a YouTube video

The clip the course uses — Star Wars droids at the Oscars, which has both R2D2
and C-3PO in it.

In [ ]:
from IPython.display import YouTubeVideo

YouTubeVideo('VkO62A_CycU')

### The course's version, with `pafy`

Stream the video straight from YouTube, run every frame through the model, and
write the annotated result to `R2D2.avi`. Press **q** in the window to stop.

Heads up: `pafy` calls `youtube_dl`, which is unmaintained, and this usually
fails against current YouTube (commonly `KeyError: 'dislike_count'`, or an
extraction error). If it does, use the `yt-dlp` cell below instead — it does the
same job. Both need `pip install` first, and network access.

In [ ]:
# pip install pafy
# pip install --upgrade youtube_dl
import pafy

url = 'https://www.youtube.com/watch?v=VkO62A_CycU'
pa = pafy.new(url)
play = pa.getbest(preftype="webm")
cap = cv2.VideoCapture(play.url)

if (cap.isOpened() == False):
    print("Unable to read video")

width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

fourcc = cv2.VideoWriter_fourcc(*'DIVX')
out = cv2.VideoWriter('R2D2.avi', fourcc, 20.0, (int(width), int(height)))

while(True):
    ret, frame = cap.read()

    if ret == True:
        frame = np.asarray(frame)
        results = tfnet2.return_predict(frame)
        new_frame = plot_box(frame, results)
        out.write(new_frame)
        cv2.imshow('frame', new_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    else:
        break

cap.release()
out.release()
cv2.destroyAllWindows()

### The same thing with `yt-dlp`

`yt-dlp` is the maintained fork of `youtube_dl`; it resolves the stream URL that
`pafy.getbest().url` used to give. Everything after that is the course's loop,
with two changes: it writes `.mp4` with `mp4v` (QuickTime cannot play a
`DIVX`/`.avi`), and it skips `cv2.imshow`, which opens a native window that does
not cooperate with a notebook kernel — set `SHOW_WINDOW = True` if you want it.

### Making the video playable in the notebook

`IPython.display.Video` renders an HTML5 `<video>` element, and browsers
decode **H.264** only. OpenCV's usual `mp4v` fourcc writes MPEG-4 Part 2
(`codec_name=mpeg4`), which writes a perfectly valid file that VLC plays
and the notebook shows as a blank player. `avc1` requests H.264 instead,
and `ensure_h264` re-encodes with ffmpeg if that ever falls back.

In [ ]:
import shutil as _shutil
import subprocess as _subprocess

# HTML5 <video> (what IPython.display.Video uses) decodes H.264 only. OpenCV's
# usual 'mp4v' fourcc writes MPEG-4 Part 2, which produces a file that saves
# fine and plays in VLC but shows a blank player in the notebook. 'avc1' asks
# FFMPEG for H.264 instead.
FOURCC = cv2.VideoWriter_fourcc(*"avc1")


def ensure_h264(path):
    """Re-encode to H.264 if the file is not already playable in a browser.

    A no-op when 'avc1' worked. Needs ffmpeg only in the fallback case.
    """
    path = str(path)
    probe = _shutil.which("ffprobe")
    if probe:
        codec = _subprocess.run(
            [probe, "-v", "error", "-select_streams", "v:0",
             "-show_entries", "stream=codec_name", "-of", "csv=p=0", path],
            capture_output=True, text=True).stdout.strip()
        if codec == "h264":
            return path
        print(f"codec is {codec!r}, re-encoding to h264 ...")

    ffmpeg = _shutil.which("ffmpeg")
    if not ffmpeg:
        print("ffmpeg not found -- install it (brew install ffmpeg) if the "
              "player below stays blank")
        return path

    tmp = path + ".h264.mp4"
    _subprocess.run(
        [ffmpeg, "-y", "-v", "error", "-i", path,
         "-c:v", "libx264", "-pix_fmt", "yuv420p",
         # libx264 requires even dimensions
         "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
         "-movflags", "+faststart", tmp],
        check=True)
    _shutil.move(tmp, path)
    return path


In [ ]:
# pip install yt-dlp
import yt_dlp

url = 'https://www.youtube.com/watch?v=VkO62A_CycU'
SHOW_WINDOW = False
MAX_FRAMES = 0        # 0 = the whole clip

with yt_dlp.YoutubeDL({'format': 'best[ext=mp4][height<=720]', 'quiet': True}) as ydl:
    info = ydl.extract_info(url, download=False)
    stream_url = info['url']

cap = cv2.VideoCapture(stream_url)

if (cap.isOpened() == False):
    print("Unable to read video")

width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

fourcc = FOURCC
VIDEO_OUT = './videos_out/R2D2-custom.mp4'
out = cv2.VideoWriter(VIDEO_OUT, fourcc, 20.0, (int(width), int(height)))

frames = 0
while(True):
    ret, frame = cap.read()

    if ret == True:
        results = tfnet2.return_predict(frame)
        new_frame = plot_box(frame, results)
        out.write(new_frame)

        frames += 1
        if frames % 25 == 0:
            print('\r{} frames'.format(frames), end='')

        if SHOW_WINDOW:
            cv2.imshow('frame', new_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        if MAX_FRAMES and frames >= MAX_FRAMES:
            break
    else:
        break

cap.release()
out.release()
ensure_h264(VIDEO_OUT)
cv2.destroyAllWindows()
print('\ndone:', frames, 'frames')

### Or from a local file

No network needed. Note that `code/5-Object_Detection/darknetv3/R2D2.avi` is
itself an *output* from an earlier darknet run — blue boxes and green labels are
burned into the pixels — so you will see two sets of boxes on it. Point
`VIDEO_IN` at an unannotated clip for a clean result.

In [ ]:
VIDEO_IN  = './code/5-Object_Detection/darknetv3/R2D2.avi'
VIDEO_OUT = './videos_out/R2D2-custom-local.mp4'

cap = cv2.VideoCapture(VIDEO_IN)
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
fps = cap.get(cv2.CAP_PROP_FPS) or 20.0

fourcc = FOURCC
out = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (int(width), int(height)))

frames = 0
while(True):
    ret, frame = cap.read()
    if ret == True:
        out.write(plot_box(frame, tfnet2.return_predict(frame)))
        frames += 1
        if frames % 25 == 0:
            print('\r{} frames'.format(frames), end='')
    else:
        break

cap.release()
out.release()
ensure_h264(VIDEO_OUT)
print('\nwrote', VIDEO_OUT, '--', frames, 'frames')

In [ ]:
from IPython.display import Video

Video(VIDEO_OUT, embed=True, width=720)

## Webcam

Three things make a notebook webcam fail on macOS, and all three are handled
below:

1. **The first frames are black.** After `VideoCapture(0)` the camera needs a
   moment to expose. On this machine the first ~5 reads come back as
   correctly-sized, entirely black images — read one immediately and you get a
   black picture and conclude the model is broken. `open_camera` polls until a
   frame actually has content.
2. **`cv2.imshow` opens a native window.** From a Jupyter kernel on macOS that
   window is often unresponsive or never appears, because the kernel is not a
   GUI app with a proper event loop. These cells draw the frames *inline* in the
   notebook output instead, so no window is involved.
3. **The camera stays locked if the loop is interrupted.** Every cell releases
   the device in a `finally`, so pressing stop and re-running works instead of
   failing with a busy device.

**Permission:** macOS must grant Camera access to the app running the kernel —
VS Code, Terminal, or whichever program you launched Jupyter from — under
*System Settings > Privacy & Security > Camera*. If it was never granted, macOS
does not raise an error; it just hands back black frames forever. That is what
the check below detects and reports.

In [ ]:
import time

from IPython.display import display, clear_output, Image as IPyImage


def open_camera(index=0, width=1280, height=720,
                settle_seconds=1.5, warmup_seconds=6.0):
    """Open the camera and return (capture, first_usable_frame).

    Two separate problems are handled here, measured on this machine:

    * Auto-exposure. The first frames are pure black (mean 0.00), then ramp
      2 -> 50 over about 1.3s before levelling off near 53. Reading one frame
      straight after VideoCapture() gives you a black image, which looks like a
      broken model rather than a camera still opening its shutter. So discard
      frames for `settle_seconds` first.
    * Missing permission. macOS raises no error when Camera access was never
      granted -- it returns correctly-sized frames that are *exactly* zero,
      forever. That is why the test below is "not pure black" rather than a
      brightness threshold: a dark room still carries sensor noise, so it
      passes, while a permission failure does not.
    """
    backend = cv2.CAP_AVFOUNDATION if sys.platform == "darwin" else cv2.CAP_ANY
    cap = cv2.VideoCapture(index, backend)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)

    if not cap.isOpened():
        cap.release()
        raise RuntimeError(
            f"cannot open camera {index}. Close any other app using it "
            "(Photo Booth, Zoom, FaceTime) and try again.")

    started = time.time()

    while time.time() - started < settle_seconds:      # let exposure settle
        cap.read()

    while time.time() - started < warmup_seconds:      # then demand real pixels
        ok, frame = cap.read()
        if ok and frame is not None and frame.mean() > 0.5:
            return cap, frame

    cap.release()
    raise RuntimeError(
        f"camera {index} opened, but every frame was pure black for "
        f"{warmup_seconds:.0f}s.\n"
        "On macOS that means Camera permission was never granted to the app "
        "running this kernel -- grant it under System Settings > Privacy & "
        "Security > Camera, then restart the kernel. macOS reports no error "
        "in this case, it simply returns black frames.")


def show_inline(frame_bgr, max_width=720):
    """Draw a BGR frame in the cell output, replacing the previous one."""
    h, w = frame_bgr.shape[:2]
    if w > max_width:
        frame_bgr = cv2.resize(frame_bgr, (max_width, int(h * max_width / w)))
    ok, buf = cv2.imencode(".jpg", frame_bgr)      # imencode wants BGR
    if ok:
        clear_output(wait=True)
        display(IPyImage(data=buf.tobytes()))

### Check the camera first

In [ ]:
# Camera self-test. Reports only metadata -- it does not display or save a frame.
try:
    cap, frame = open_camera()
    cap.release()
    print(f"camera OK  frame={frame.shape}  brightness={frame.mean():.1f}")
except RuntimeError as exc:
    print("camera NOT usable:\n", exc)

### Detection from the camera

In [ ]:
RUN_WEBCAM = False      # set True to start
DURATION = 20           # seconds; or press the stop button

if RUN_WEBCAM:
    cap, frame = open_camera()
    frames = 0
    started = time.time()
    try:
        while time.time() - started < DURATION:
            ok, frame = cap.read()
            if not ok:
                break

            results = tfnet2.return_predict(frame)
            for result in results:
                tl = (result['topleft']['x'], result['topleft']['y'])
                br = (result['bottomright']['x'], result['bottomright']['y'])
                text = '{}: {:.0f}%'.format(result['label'], result['confidence'] * 100)
                frame = cv2.rectangle(frame, tl, br, (0, 0, 255), 5)
                frame = cv2.putText(frame, text, tl, cv2.FONT_ITALIC, 1, (255, 255, 255), 2)

            show_inline(frame)
            frames += 1
    except KeyboardInterrupt:
        print("interrupted")
    finally:
        cap.release()                 # always release, or the device stays busy
        cv2.destroyAllWindows()

    elapsed = time.time() - started
    print(f"stopped: {frames} frames in {elapsed:.1f}s ({frames / max(elapsed, 1e-6):.1f} FPS)")
else:
    print("Set RUN_WEBCAM = True to start the camera.")